# Structure, Diffraction, And Visualization Pipeline

This notebook shows the architectural point of the new features: one `Phase` can
feed structure visualization, powder XRD, and SAED workflows without redefining the
crystallographic semantics at each step.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("A1", "Cu", np.array([0.0, 0.0, 0.0])),
            AtomicSite("A2", "Cu", np.array([0.0, 0.5, 0.5])),
            AtomicSite("A3", "Cu", np.array([0.5, 0.0, 0.5])),
            AtomicSite("A4", "Cu", np.array([0.5, 0.5, 0.0])),
        ),
    )
    phase = Phase(
        "fcc_demo",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

scene = build_crystal_scene(phase, repeats=(2, 2, 2), plane_hkls=((1, 1, 1),))
xrd = generate_xrd_pattern(phase, broadening_fwhm_deg=0.16, max_index=6)
saed = generate_saed_pattern(
    phase,
    ZoneAxis(indices=np.array([1, 1, 0]), phase=phase),
    camera_constant_mm_angstrom=180.0,
    max_index=5,
    max_g_inv_angstrom=3.5,
)

print("scene atoms:", len(scene.atoms))
print("xrd reflections:", len(xrd.reflections))
print("saed spots:", len(saed.spots))


In [ ]:
xrd_figure = plot_xrd_pattern(xrd, theme="journal")
saed_figure = plot_saed_pattern(saed, theme="dark")
crystal_figure = plot_crystal_structure_3d(scene, projection="persp")
xrd_figure


In [ ]:
saed_figure


In [ ]:
crystal_figure
